# [센서 실습 16] pH 센서 – PH-4502C + 유리전극

pH 측정 모듈 **PH-4502C**의 출력을 16비트 ADC로 읽고, **pH 7·pH 4 완충용액 2점 보정**으로 변환식을 직접 구한 뒤, 수조의 산성화를 감지합니다.

**이 노트북의 흐름**
1. 센서의 원리와 부품 사양
2. 라즈베리파이 연결 방법
3. 실제 센서를 읽는 코드 (라즈베리파이용)
4. 장비가 없을 때 쓰는 **가상 센서** (같은 코드로 실행)
5. 수집 → 판정 → 시각화 → 저장·IoT 전송

> 장비가 없어도 `USE_REAL_SENSOR = False` 상태 그대로 위에서부터 `Shift + Enter`로 모두 실행됩니다.

## 1. 센서의 원리
pH는 물이 산성인지 알칼리성인지를 나타내는 값입니다 (0~14, 7이 중성).
- **유리 전극**은 수소 이온 농도에 따라 아주 작은 전압을 냄: 25°C에서 pH 1당 약 **59mV** (네른스트 식)
- 전극 신호는 매우 약해 모듈(PH-4502C)이 **증폭**하여 0~5V로 내보냄 → pH 7에서 약 2.5V, 산성일수록 전압이 높아짐
- 전극마다 특성이 다르고 시간이 지나면 변하므로 **완충용액으로 보정**해야 함

$$\text{기울기} = \frac{7 - 4}{V_{7} - V_{4}}, \qquad \text{pH} = \text{기울기} \times (V - V_7) + 7$$

### 부품 사양
| 항목 | 내용 |
|---|---|
| 측정 범위 | pH 0 ~ 14 |
| 정확도 | ±0.1 pH (25°C, 보정 후) |
| 출력 | 아날로그 0 ~ 5V (pH 7 ≈ 2.5V) |
| 전원 | 5V |
| 보정 | pH 7.0, 4.0 (또는 10.0) 완충용액 |

## 2. 라즈베리파이 연결 방법
**준비물**: PH-4502C 모듈 + pH 유리전극(BNC), ADS1115, 저항 10kΩ·20kΩ, pH 7·pH 4 완충용액, 증류수

| 연결할 곳 | 라즈베리파이 | 설명 |
|---|---|---|
| ADS1115 VDD | 3.3V (1번 핀) |  |
| ADS1115 GND | GND (6번 핀) |  |
| ADS1115 SCL | GPIO3 / SCL (5번 핀) | I2C 클록 |
| ADS1115 SDA | GPIO2 / SDA (3번 핀) | I2C 데이터 |
| ADS1115 ADDR | GND | I2C 주소 0x48 |
| PH-4502C V+ / G | 5V / GND |  |
| PH-4502C Po → 10kΩ → ADS1115 A0 | A0 ↔ GND 사이 20kΩ | 5V 출력을 2/3로 낮춤 |

> 💡 전극은 보관액(KCl)에 담가 두어야 하며, 다른 용액으로 옮길 때마다 증류수로 헹굽니다.

**인터페이스 활성화**
- **I2C**: `sudo raspi-config` → Interface Options → **I2C** → Enable, 연결 확인: `i2cdetect -y 1`

**라이브러리 설치 (라즈베리파이 터미널)**
```bash
pip3 install adafruit-circuitpython-ads1x15
```

## 3. 공통 준비 – 한글 폰트와 라이브러리

In [ ]:
# Colab에서 그래프의 한글이 깨지지 않도록 나눔고딕 폰트를 설정합니다. (처음 한 번 실행)
import os, subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(font_path):
    subprocess.run("apt-get -qq -y install fonts-nanum", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
import time, json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 4. 실제 센서 읽기 (라즈베리파이용)
### 4-1. 원시값 → 물리량 변환식 (실제·가상 공통)
완충용액에서 읽은 두 전압으로 **기울기와 기준점**을 구하는 보정 함수와, 전압 → pH 변환 함수입니다.

In [ ]:
DIVIDER = 1.5

def calibrate(v_ph7, v_ph4):
    """2점 보정: pH 7과 pH 4 완충용액에서 읽은 센서 전압 → (기울기, V7)"""
    slope = (7.0 - 4.0) / (v_ph7 - v_ph4)
    return slope, v_ph7

SLOPE, V7 = calibrate(v_ph7=2.50, v_ph4=3.03)     # ← 각자 측정한 값으로 바꾸기
print(f"보정 결과: 기울기 {SLOPE:.3f} pH/V (1 pH당 {1000 / abs(SLOPE):.0f} mV)")

def voltage_to_ph(v_sensor):
    return round(SLOPE * (v_sensor - V7) + 7.0, 2)

for v in (2.50, 2.32, 2.85, 3.03):
    print(f"{v:.2f}V → pH {voltage_to_ph(v)}")

### 4-2. 실제 센서 클래스
ADS1115 A0 채널 전압을 10회 평균하여 읽습니다. (pH 전극은 잡음이 많음)

> 이 셀은 **클래스를 정의만** 합니다. 하드웨어 라이브러리는 객체를 만들 때(`__init__`) 불러오므로 Colab에서 실행해도 오류가 나지 않습니다. 실제 동작은 라즈베리파이에서 `USE_REAL_SENSOR = True`로 해야 합니다.

In [ ]:
class RealPH4502C:
    def __init__(self):
        import board
        import adafruit_ads1x15.ads1115 as ADS
        from adafruit_ads1x15.analog_in import AnalogIn
        ads = ADS.ADS1115(board.I2C()); ads.gain = 1
        self.ch = AnalogIn(ads, ADS.P0)

    def read(self):
        vs = []
        for _ in range(10):
            vs.append(self.ch.voltage); time.sleep(0.02)
        v_sensor = float(np.mean(vs)) * DIVIDER
        return {"v_sensor": round(v_sensor, 4), "ph": voltage_to_ph(v_sensor)}

## 5. 장비가 없을 때 – 가상 센서
### 왜 가상 센서를 쓰나요?
실제 센서의 값은 다음 순서로 만들어집니다.

```
① 전기 신호 (전압, 펄스, 0/1)  →  ② 원시값 raw (ADC 숫자, 바이트, 문자열)  →  ③ 변환식  →  ④ 물리량 (°C, ppm, cm …)
```
가상 센서는 ①을 건너뛰고 **②원시값을 실제 센서와 똑같은 형식으로** 만들어 줍니다.
따라서 ③변환식과 그 뒤의 판정·시각화·저장 코드는 실제 장비와 가상 장비에서 **완전히 같은 코드**를 사용합니다.

```
[장비 있음]  센서 ──배선──▶ 라즈베리파이 ──▶ Real….read()  ─┐
                                                             ├──▶ 변환식 → 판정 → 그래프 → 저장/전송
[장비 없음]  가정한 상황(시나리오) ──▶ Virtual….read() ──────┘
```
나중에 장비를 연결하면 `USE_REAL_SENSOR = True` **한 줄만** 바꾸면 됩니다.

### 가상 값은 이렇게 정했습니다
| 항목 | 가정한 값 | 근거 |
|---|---|---|
| 상황 | 물고기 수조, 24시간 15분 간격 | 수질 감시 |
| 평상시 | pH 7.2, 낮에 +0.15 (수초 광합성으로 CO₂ 감소) | 수조의 하루 변화 |
| 오염 | 10시부터 산성 물질 유입 → 최저 약 pH 5.6 | 사고 상황 |
| 중화 | 16시 중화제 투입 → 서서히 회복 | 조치 |
| 전극 특성 | pH 7에서 2.50V, 1 pH당 −0.177V, 잡음 ±3mV | 보정값과 같은 특성 |

In [ ]:
class VirtualPH4502C:
    def __init__(self, dt=900, seed=16):
        self.dt, self.t = dt, 0.0
        self.rng = np.random.default_rng(seed)

    def read(self):
        h = self.t / 3600
        ph = 7.2 + 0.15 * np.sin(np.pi * (h - 8) / 12) * (6 <= h % 24 <= 20)
        if 10 <= h < 16:
            ph -= 1.6 * (1 - np.exp(-(h - 10) / 1.5))                  # 산성화
        elif h >= 16:
            ph -= 1.6 * (1 - np.exp(-6 / 1.5)) * np.exp(-(h - 16) / 1.5)   # 중화
        v_sensor = 2.50 - (ph - 7.0) * (0.53 / 3) + self.rng.normal(0, 0.003)
        v_adc = round(v_sensor / DIVIDER, 5)
        self.t += self.dt
        return {"v_sensor": round(v_adc * DIVIDER, 4), "ph": voltage_to_ph(v_adc * DIVIDER)}

In [ ]:
# 가상 센서 동작 확인 – read()를 3번 호출
test = VirtualPH4502C(dt=900)
for _ in range(3):
    print(test.read())

## 6. 센서 선택과 데이터 수집
15분 간격으로 24시간 측정합니다.

- 실제 장비는 `time.sleep(INTERVAL)`으로 측정 간격만큼 기다립니다.
- 가상 센서는 기다리지 않고 시간을 건너뛰므로, 몇 시간·며칠 분량의 데이터도 바로 만들어집니다.

In [ ]:
USE_REAL_SENSOR = False      # ← 라즈베리파이에 센서를 연결했다면 True 로 변경
N_SAMPLES = 96               # 측정 횟수
INTERVAL = 900               # 측정 간격(초)

sensor = RealPH4502C() if USE_REAL_SENSOR else VirtualPH4502C(dt=INTERVAL)

records = []
for i in range(N_SAMPLES):
    value = sensor.read()
    records.append({"time_s": round(i * INTERVAL, 4), **value})
    if USE_REAL_SENSOR:
        time.sleep(INTERVAL)

df = pd.DataFrame(records)
print(f"{len(df)}개 측정 완료 (총 {N_SAMPLES * INTERVAL / 60:.1f}분 분량)")
df.head()

In [ ]:
df.describe().round(3)

## 7. 판정 – 이벤트 감지
물고기가 살기 좋은 범위를 **pH 6.5 ~ 8.0**으로 보고, 6.0 미만이면 위험으로 판정합니다.

In [ ]:
df["hour"] = df["time_s"] / 3600
df["state"] = np.where(df["ph"] < 6.0, "위험(산성)", np.where(df["ph"] < 6.5, "주의", np.where(df["ph"] > 8.0, "주의(알칼리)", "정상")))
prev = None
for _, r in df.iterrows():
    if r["state"] != prev:
        print(f"{r['hour']:5.2f}시  pH {r['ph']:.2f}  → {r['state']}"); prev = r["state"]
print(f"\n최저 pH {df['ph'].min():.2f} ({df.loc[df['ph'].idxmin(), 'hour']:.1f}시)")

## 8. 시각화

In [ ]:
plt.figure(figsize=(10, 4.5))
plt.axhspan(6.5, 8.0, color="0.92", label="적정 범위 6.5~8.0")
plt.plot(df["hour"], df["ph"], "k-o", ms=3, lw=1.2, label="pH")
plt.axhline(6.0, color="gray", ls="--", label="위험 6.0")
plt.ylim(5, 8.5); plt.xticks(range(0, 25, 3))
plt.title("PH-4502C 수조 pH"); plt.xlabel("시각 (시)"); plt.ylabel("pH")
plt.grid(alpha=0.3); plt.legend(loc="lower left"); plt.show()

## 9. 저장과 IoT 전송
측정값은 CSV 파일로 저장하고, IoT 서버로 보낼 때는 보통 **JSON** 형식의 메시지를 **MQTT** 프로토콜로 전송합니다.
(Colab에서는 왼쪽 📁 메뉴에서 저장된 CSV를 내려받을 수 있습니다.)

In [ ]:
csv_name = "pH센서_data.csv"
df.to_csv(csv_name, index=False, encoding="utf-8-sig")
print("CSV 저장:", csv_name)

def to_jsonable(v):
    if isinstance(v, (bool, np.bool_)): return bool(v)
    if isinstance(v, (int, np.integer)): return int(v)
    if isinstance(v, (float, np.floating)): return None if np.isnan(v) else round(float(v), 3)
    return str(v)

last = df.iloc[-1].to_dict()
payload = json.dumps({"device": "rpi-01", "sensor": "PH-4502C + 유리전극",
                      "time": datetime.now().isoformat(timespec="seconds"),
                      "values": {k: to_jsonable(v) for k, v in last.items()}}, ensure_ascii=False)
print("IoT 서버로 보낼 메시지(JSON):")
print(payload)

# ▼ 실제로 MQTT 브로커에 보내려면 (pip install paho-mqtt) 주석을 해제하세요.
# import paho.mqtt.client as mqtt
# client = mqtt.Client()
# client.connect("broker.hivemq.com", 1883)          # 공개 테스트 브로커
# client.publish("myclass/rpi-01/ph", payload)
# client.disconnect()

## 직접 해 보기
- 보정 값을 `calibrate(2.48, 3.05)`로 바꾸면 같은 전압의 pH가 얼마나 달라지나요?
- pH 7과 pH 10 완충용액으로 보정하려면 `calibrate` 함수를 어떻게 바꿔야 할까요?
- pH가 6.0 아래로 내려가면 중화제 펌프를 켜는 제어를 반복문에 추가해 보세요.